In [ ]:
import qlib
from qlib.config import REG_CN

qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region=REG_CN
)

print("Qlib initialized")

In [ ]:
from qlib.contrib.data.handler import Alpha158
from qlib.data.dataset import DatasetH
from qlib.contrib.model.gbdt import LGBModel

from qlib.contrib.strategy import TopkDropoutStrategy
from qlib.backtest import backtest
from qlib.contrib.evaluate import risk_analysis

In [ ]:
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2015-01-01",
            "end_time": "2020-12-31",
            "fit_start_time": "2015-01-01",
            "fit_end_time": "2018-12-31",
            "instruments": "csi500",
        },
    },
    segments={
        "train": ("2015-01-01", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
)

In [ ]:
model = LGBModel(
    loss="mse",
    learning_rate=0.05,
    num_leaves=210,
    n_estimators=200
)

model.fit(dataset)

In [ ]:
pred = model.predict(dataset)

pred.head()

In [ ]:
strategy = TopkDropoutStrategy(
    signal=pred,
    topk=50,
    n_drop=5
)

In [ ]:
from qlib.backtest.executor import SimulatorExecutor

executor = SimulatorExecutor(
    time_per_step="day",
    generate_portfolio_metrics=True
)

In [ ]:
report, positions = backtest(
    executor=executor,
    strategy=strategy,
    start_time="2020-01-01",
    end_time="2020-12-31",
    account=10000000,
    benchmark="SH000300"
)

In [ ]:
report


In [ ]:
report_df = report["1day"][0]

analysis = risk_analysis(report_df["return"] - report_df["bench"])
analysis

In [ ]:
import matplotlib.pyplot as plt

strategy_curve = (1 + report_df["return"]).cumprod()
bench_curve = (1 + report_df["bench"]).cumprod()

plt.figure(figsize=(10,5))
plt.plot(strategy_curve, label="Strategy")
plt.plot(bench_curve, label="Benchmark")
plt.legend()
plt.title("Strategy vs Benchmark")
plt.show()

In [ ]:
equity = (1 + report_df["return"]).cumprod()

drawdown = equity / equity.cummax() - 1

plt.figure(figsize=(10,4))
drawdown.plot()
plt.title("Drawdown")
plt.show()

In [ ]:
report_df["turnover"].mean()